# Estudo de caso 2.4 - Previsão de salários II

Configuração do *notebook*:

Sincronize sua conta do Google. Para isso, siga o link que aparece na saída da seguinte célula uma vez executada. Copie o código que aparece na tela e insira-o na saída da célula. Assim que visualizar a mensagem: `Google Drive sincronizado com sucesso!`poderá continuar executando o restante das células.

In [ ]:
!pip install rpy2==3.5.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.7/201.7 kB 10.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rpy2: filename=rpy2-3.5.1-cp310-cp310-linux_x86_64.whl size=318095 sha256=ae942ceddf19cf184256677cc4732afe7a9fdef53913fa27c9dc5ad8bbd9774a
  Stored in directory: /root/.cache/pip/wheels/73/a6/ff/4e75dd1ce1cfa2b9a670cbccf6a1e41c553199e9b25f05d953
Successfully built rpy2
  Attempting uninstall: rpy2
    Found existing installation: rpy2 3.5.5
    Uninstalling rpy2-3.5.5:
      Successfully uninstalled rpy2-3.5.5


In [ ]:
from google.colab import auth
auth.authenticate_user()

from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)
data_drop = drive.CreateFile({'id':'1uQj3lEsilNJzwxWNn-PBM4exOUTBDFy7'})
data_drop.GetContentFile('wage2015.Rdata')

print('Google Drive sincronizado com sucesso')

Google Drive sincronizado com sucesso


In [ ]:
%load_ext rpy2.ipython

Instalar e importar bibliotecas: (ignorar resultados a não ser que não apareça a frase: `Bibliotecas instaladas com sucesso!`)

In [ ]:
%%R --noreturn
# Instalar bibliotecas
install.packages("hdm")
install.packages("randomForest")
install.packages("glmnet")
install.packages("nnet")
install.packages("rpart")
install.packages("gbm")
install.packages("rpart.plot")

cat('\nBibliotecas instaladas com sucesso!')

(as ‘lib’ is unspecified)

















































	‘/tmp/RtmpZ1gn8y/downloaded_packages’

(as ‘lib’ is unspecified)







	‘/tmp/RtmpZ1gn8y/downloaded_packages’

(as ‘lib’ is unspecified)







	‘/tmp/RtmpZ1gn8y/downloaded_packages’

(as ‘lib’ is unspecified)







	‘/tmp/RtmpZ1gn8y/downloaded_packages’

(as ‘lib’ is unspecified)







	‘/tmp/RtmpZ1gn8y/downloaded_packages’

(as ‘lib’ is unspecified)







	‘/tmp/RtmpZ1gn8y/downloaded_packages’

(as ‘lib’ is unspecified)







	‘/tmp/RtmpZ1gn8y/downloaded_packages’




Bibliotecas instaladas com sucesso!

In [ ]:
%%R
# Carregar bibliotecas
library(hdm)
library(randomForest)
library(glmnet)
library(nnet)
options(warn=-1)
library(rpart)
library(gbm)
library(rpart.plot)

cat('\nBibliotecas importadas com sucesso!')


Bibliotecas importadas com sucesso!

## Dados


In [ ]:
%%R
# Carregar o banco de dados
load(file="wage2015.Rdata")

# Ver as variáveis do banco de dados
class(data)
str(data)

# Mostrar dimensões do banco de dados
dims  <- dim(data)
cat('\nDimensões do banco de dados:',toString(dims),'\n',fill = TRUE)

'data.frame':	12697 obs. of  37 variables:
 $ wage    : num  9.62 48.08 11.06 11.54 13.94 ...
 $ lwage   : num  2.26 3.87 2.4 2.45 2.63 ...
 $ sex     : num  0 1 1 1 0 0 0 0 0 1 ...
 $ white   : num  1 1 1 1 1 1 0 1 1 1 ...
 $ black   : num  0 0 0 0 0 0 0 0 0 0 ...
 $ hisp    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ shs     : num  0 0 0 0 0 0 0 0 0 0 ...
 $ hsg     : num  0 0 1 0 0 0 0 0 1 1 ...
 $ scl     : num  0 0 0 1 0 0 0 0 0 0 ...
 $ clg     : num  1 1 0 0 0 1 1 1 0 0 ...
 $ mw      : num  0 0 0 0 0 0 0 0 0 0 ...
 $ so      : num  0 0 0 0 0 0 0 0 0 0 ...
 $ we      : num  0 0 0 0 0 0 0 0 0 0 ...
 $ union   : num  0 0 0 0 0 1 0 0 0 0 ...
 $ vet     : num  0 0 0 0 0 0 0 0 0 0 ...
 $ cent    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ ncent   : num  0 0 0 0 0 0 0 0 0 0 ...
 $ fam1    : num  0 1 1 1 1 1 1 1 1 1 ...
 $ fam2    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ fam3    : num  1 0 0 0 0 0 0 0 0 0 ...
 $ child   : num  0 0 0 0 0 0 0 0 0 0 ...
 $ fborn   : num  0 0 0 0 0 0 1 0 0 0 ...
 $ cit     : num  1 

Separação dos dados em um conjunto de treinamento e um de teste

In [ ]:
%%R
# Gerador de números aleatórios
set.seed(1)
# Conjunto de índices de treinamento
training <- sample(nrow(data), nrow(data)*(1/2), replace=FALSE)
# Conjunto de treinamento
datause <- data[training,]
# Conjunto de teste
dataout <- data[-training,]

## Metodologia

### Definição de modelos (básico e flexível)

In [ ]:
%%R
# Variáveis de controle lineares. Usar estas variáveis de controle para métodos baseados em árvores de regressão
x <- "sex+white+black+hisp+shs+hsg+scl+clg+mw+so+we+union+vet+cent+ncent+fam1+fam2+fam3+child+\
fborn+cit+school+pens+fsize10+fsize100+health+age+exp1+occ2+ind2"

# Variáveis de controle quadráticas (especificação flexível). Usar estas variáveis de controle para métodos lineares
xL <- "(sex+white+black+hisp+shs+hsg+scl+clg+mw+so+we+union+vet+cent+ncent+fam1+fam2+fam3+child+\
fborn+cit+school+pens+fsize10+fsize100+health+age+exp1+exp2+exp3+exp4+occ2+ind2)^2"

# variável de resultado: log wage (logaritmo do salário)
y  <- "lwage"

# Modelo linear: Especificação quadrática
formL <- as.formula(paste(y, "~", xL))
# Modelo linear: Especificação linear
form  <- as.formula(paste(y, "~", x))

In [ ]:
%%R
# "-1" : não incluir a constante no modelo
# x,y TRUE: retornar a matriz de covariáveis/vetor de resultados
# Execute estas regressões lineares para usar suas variáveis de resultado (fituse$y) e covariáveis (fituse$y)
# um truque para extrair as variáveis x e y de uma fórmula

fituseL    <- lm(paste(y, "~", xL, "-1"), datause, x=TRUE, y=TRUE)
fitoutL    <- lm(paste(y, "~", xL, "-1"), dataout, x=TRUE, y=TRUE)
fituse     <- lm(paste(y, "~", x,  "-1"), datause, x=TRUE, y=TRUE)
fitout     <- lm(paste(y, "~", x,  "-1"), dataout, x=TRUE, y=TRUE)

### Treinamento com métodos lineares e não lineares

In [ ]:
%%R
start_time <- Sys.time()

# Regressão linear modelo simples
fit.lm      <- lm(form, datause)

end_time <- Sys.time()
sprintf('Tempo de treinamento OLS: %.2f s',difftime(end_time,start_time,units = 'secs'))

[1] "Tempo de treinamento OLS: 0.03 s"


In [ ]:
%%R
start_time <- Sys.time()

# Regressão linear modelo flexível
fit.lm2     <- lm(formL, datause)

end_time <- Sys.time()
sprintf('Tempo de treinamento OLS (flexível): %.2f s',difftime(end_time,start_time,units = 'secs'))

[1] "Tempo de treinamento OLS (flexível): 17.03 s"


In [ ]:
%%R
start_time <- Sys.time()

#Lasso
fit.rlasso   <- rlasso(form, datause, post=FALSE)

end_time <- Sys.time()
sprintf('Tempo de treinamento Lasso: %.2f s',difftime(end_time,start_time,units = 'secs'))

[1] "Tempo de treinamento Lasso: 0.55 s"


In [ ]:
%%R
start_time <- Sys.time()

#Pós-Lasso
fit.rlasso2  <- rlasso(form, datause, post=TRUE)

end_time <- Sys.time()
sprintf('Tempo de treinamento Pós-Lasso: %.2f s',difftime(end_time,start_time,units = 'secs'))

[1] "Tempo de treinamento Pós-Lasso: 0.70 s"


Configuração de regressões penalizadas:

* alpha=1: norma 1 (Lasso), alpha=0: norma 2 (Ridge)
* alpha = 0.5 : ambas penalidades (Elastic Net)

* post = FALSE: sem voltar a executar mínimos quadrados nas variáveis selecionadas

In [ ]:
%%R
start_time <- Sys.time()

#Lasso com Validação Cruzada (VC)
fit.lasso    <- cv.glmnet(fituse$x, fituse$y, family="gaussian", alpha=1)

end_time <- Sys.time()
sprintf('Tempo de treinamento Lasso com VC: %.2f s',difftime(end_time,start_time,units = 'secs'))

[1] "Tempo de treinamento Lasso com VC: 0.53 s"


In [ ]:
%%R
start_time <- Sys.time()

#Ridge com VC
fit.ridge    <- cv.glmnet(fituse$x, fituse$y, family="gaussian", alpha=0)

end_time <- Sys.time()
sprintf('Tempo de treinamento Ridge com VC: %.2f s',difftime(end_time,start_time,units = 'secs'))

[1] "Tempo de treinamento Ridge com VC: 0.30 s"


In [ ]:
%%R
start_time <- Sys.time()

#Elastic Net com VC
fit.elnet    <- cv.glmnet(fituse$x, fituse$y, family="gaussian", alpha=.5)

end_time <- Sys.time()
sprintf('Tempo de treinamento Elastic Net com VC: %.2f s',difftime(end_time,start_time,units = 'secs'))

[1] "Tempo de treinamento Elastic Net com VC: 0.68 s"


ATENÇÃO: a célula seguinte pode levar cerca de 4 minutos para ser executada

In [ ]:
%%R
start_time <- Sys.time()

#Lasso (flexível)
fit.rlassoL  <- rlasso(formL, datause, post=FALSE)

end_time <- Sys.time()
sprintf('Tempo de treinamento Lasso (flexível): %.2f min',difftime(end_time,start_time,units = 'mins'))

[1] "Tempo de treinamento Lasso (flexível): 3.52 min"


ATENÇÃO: a célula seguinte pode levar cerca de 2 minutos para ser executada

In [ ]:
%%R
start_time <- Sys.time()

#Pós-Lasso (flexível)
fit.rlasso2L <- rlasso(formL, datause, post=TRUE)

end_time <- Sys.time()
sprintf('Tempo de treinamento Pós-Lasso (flexível): %.2f mins',difftime(end_time,start_time,units = 'mins'))

[1] "Tempo de treinamento Pós-Lasso (flexível): 1.31 mins"


ATENÇÃO: a célula seguinte pode levar cerca de 14 minutos para ser executada

In [ ]:
%%R
start_time <- Sys.time()

#Lasso com VC (flexível)
fit.lassoL   <- cv.glmnet(fituseL$x, fituseL$y, family="gaussian", alpha=1)

end_time <- Sys.time()
sprintf('Tempo de treinamento Lasso com VC (flexível): %.2f mins',difftime(end_time,start_time,units = 'mins'))

[1] "Tempo de treinamento Lasso com VC (flexível): 7.08 mins"


ATENÇÃO: a célula seguinte pode levar cerca de 2 minutos para ser executada

In [ ]:
%%R
start_time <- Sys.time()

#Ridge com VC (flexível)
fit.ridgeL   <- cv.glmnet(fituseL$x, fituseL$y, family="gaussian", alpha=0)

end_time <- Sys.time()
sprintf('Tempo de treinamento Ridge com VC (flexível): %.2f mins',difftime(end_time,start_time,units = 'mins'))

[1] "Tempo de treinamento Ridge com VC (flexível): 1.06 mins"


ATENÇÃO: a célula seguinte pode levar cerca de 13 minutos para ser executada

In [ ]:
%%R
start_time <- Sys.time()

#Elastic Net com VC (flexível)
fit.elnetL   <- cv.glmnet(fituseL$x, fituseL$y, family="gaussian", alpha=.5)

end_time <- Sys.time()
sprintf('Tempo de treinamento Elastic Net com VC (flexível): %.2f mins',difftime(end_time,start_time,units = 'mins'))

[1] "Tempo de treinamento Elastic Net com VC (flexível): 6.43 mins"


ATENÇÃO: a célula seguinte pode levar cerca de 4 minutos para ser executada

In [ ]:
%%R
start_time <- Sys.time()

#Random Forest
fit.rf       <- randomForest(form, ntree=2000, nodesize=5, data=datause)

end_time <- Sys.time()
sprintf('Tempo de treinamento Random Forest: %.2f mins',difftime(end_time,start_time,units = 'mins'))

[1] "Tempo de treinamento Random Forest: 4.15 mins"


In [ ]:
%%R
start_time <- Sys.time()

#Boosting Trees
fit.boost    <- gbm(form, data=datause, distribution= "gaussian", bag.fraction = .5, interaction.depth=2, n.trees=1000, shrinkage=.01)
best.boost   <- gbm.perf(fit.boost, plot.it = FALSE)

end_time <- Sys.time()
sprintf('Tempo de treinamento Boosting Trees: %.2f s',difftime(end_time,start_time,units = 'secs'))

[1] "Tempo de treinamento Boosting Trees: 4.68 s"


In [ ]:
%%R
start_time <- Sys.time()

#Regression Trees (Árvores de regressão, neste caso podadas)
fit.trees    <- rpart(form, datause)
bestcp       <- trees$cptable[which.min(trees$cptable[,"xerror"]),"CP"]
fit.prunedtree <- prune(fit.trees,cp=bestcp)

end_time <- Sys.time()
sprintf('Tempo de treinamento Árvore podada: %.2f s',difftime(end_time,start_time,units = 'secs'))

[1] "Tempo de treinamento Árvore podada: 0.38 s"


In [ ]:
%%R
start_time <- Sys.time()

#Rede neural
fit.nnet     <- nnet(form, datause, size=5,  maxit=1000, MaxNWts=100000, decay=0.01, linout = TRUE, trace=FALSE)   # ajuste simples de uma rede neural

end_time <- Sys.time()
sprintf('Tempo de treinamento Rede Neural: %.2f s',difftime(end_time,start_time,units = 'secs'))

[1] "Tempo de treinamento Rede Neural: 13.60 s"


Cálculo das previsões fora da amostra:


In [ ]:
%%R

yhat.lm       <- predict(fit.lm,        newdata= dataout)
yhat.lm2      <- predict(fit.lm2,       newdata= dataout)
yhat.rlasso   <- predict(fit.rlasso,    newdata= dataout)
yhat.rlasso2  <- predict(fit.rlasso2,   newdata= dataout)
yhat.lasso    <- predict(fit.lasso,     newx   = fitout$x)
yhat.ridge    <- predict(fit.ridge,     newx   = fitout$x)
yhat.elnet    <- predict(fit.elnet,     newx   = fitout$x)
yhat.rlassoL  <- predict(fit.rlassoL,   newdata= dataout)
yhat.rlasso2L <- predict(fit.rlasso2L,  newdata= dataout)
yhat.lassoL   <- predict(fit.lassoL,    newx   = fitoutL$x)
yhat.ridgeL   <- predict(fit.ridgeL,    newx   = fitoutL$x)
yhat.elnetL   <- predict(fit.elnetL,    newx   = fitoutL$x)
yhat.rf       <- predict(fit.rf,        newdata= dataout)
yhat.boost    <- predict(fit.boost,     newdata= dataout, n.trees=best.boost)
yhat.pt       <- predict(fit.prunedtree,newdata= dataout)
yhat.nnet     <- predict(fit.nnet,      newdata= dataout)

Cálculo do EQM para cada modelo:

In [ ]:
%%R
y.test       = dataout$lwage
MSE.lm       = summary(lm((y.test-yhat.lm)^2~1))$coef[1:2]
MSE.lm2      = summary(lm((y.test-yhat.lm2)^2~1))$coef[1:2]
MSE.rlasso   = summary(lm((y.test-yhat.rlasso)^2~1))$coef[1:2]
MSE.rlasso2  = summary(lm((y.test-yhat.rlasso2)^2~1))$coef[1:2]
MSE.lasso    = summary(lm((y.test-yhat.lasso)^2~1))$coef[1:2]
MSE.ridge    = summary(lm((y.test-yhat.ridge)^2~1))$coef[1:2]
MSE.elnet    = summary(lm((y.test-yhat.elnet)^2~1))$coef[1:2]
MSE.rlassoL  = summary(lm((y.test-yhat.rlassoL)^2~1))$coef[1:2]
MSE.rlasso2L = summary(lm((y.test-yhat.rlasso2L)^2~1))$coef[1:2]
MSE.lassoL   = summary(lm((y.test-yhat.lassoL)^2~1))$coef[1:2]
MSE.ridgeL   = summary(lm((y.test-yhat.ridgeL)^2~1))$coef[1:2]
MSE.elnetL   = summary(lm((y.test-yhat.elnetL)^2~1))$coef[1:2]
MSE.rf       = summary(lm((y.test-yhat.rf)^2~1))$coef[1:2]
MSE.boost    = summary(lm((y.test-yhat.boost)^2~1))$coef[1:2]
MSE.pt       = summary(lm((y.test-yhat.pt)^2~1))$coef[1:2]
MSE.nnet     = summary(lm((y.test-yhat.nnet)^2~1))$coef[1:2]

Salvar os resultados em uma tabela:

In [ ]:
%%R
table          <- matrix(0, 16, 3)
table[1,1:2]   <- MSE.lm
table[2,1:2]   <- MSE.lm2
table[3,1:2]   <- MSE.rlasso
table[4,1:2]   <- MSE.rlassoL
table[5,1:2]   <- MSE.rlasso2
table[6,1:2]   <- MSE.rlasso2L
table[7,1:2]   <- MSE.lasso
table[8,1:2]   <- MSE.lassoL
table[9,1:2]   <- MSE.ridge
table[10,1:2]  <- MSE.ridgeL
table[11,1:2]  <- MSE.elnet
table[12,1:2]  <- MSE.elnetL
table[13,1:2]  <- MSE.rf
table[14,1:2]  <- MSE.boost
table[15,1:2]  <- MSE.pt
table[16,1:2]  <- MSE.nnet

table[1,3]   <- 1-MSE.lm[1]/var(y.test)
table[2,3]   <- 1-MSE.lm2[1]/var(y.test)
table[3,3]   <- 1-MSE.rlasso[1]/var(y.test)
table[4,3]   <- 1-MSE.rlassoL[1]/var(y.test)
table[5,3]   <- 1-MSE.rlasso2[1]/var(y.test)
table[6,3]   <- 1-MSE.rlasso2L[1]/var(y.test)
table[7,3]   <- 1-MSE.lasso[1]/var(y.test)
table[8,3]   <- 1-MSE.lassoL[1]/var(y.test)
table[9,3]   <- 1-MSE.ridge[1]/var(y.test)
table[10,3]  <- 1-MSE.ridgeL[1]/var(y.test)
table[11,3]  <- 1-MSE.elnet[1]/var(y.test)
table[12,3]  <- 1-MSE.elnetL[1]/var(y.test)
table[13,3]  <- 1-MSE.rf[1]/var(y.test)
table[14,3]  <- 1-MSE.boost[1]/var(y.test)
table[15,3]  <- 1-MSE.pt[1]/var(y.test)
table[16,3]  <- 1-MSE.nnet[1]/var(y.test)


# Atribuir nomes a columnas e filas
colnames(table)<- c("EQM", "Erro Est. do EQM", "R^2")
rownames(table)<- c("Mínimos Quadrados", "Mínimos Quadrados (flexível)", "Lasso", "Lasso (flexível)", "Pós-Lasso",  "Pós-Lasso (flexível)",
                    "Lasso com VC", "Lasso com VC (flexível)", "Ridge com VC", "Ridge com VC (flexível)", "Elastic Net com VC", "Elastic Net com VC (Flexível)",
                    "Random Forest","Boosted Trees", "Árvore podada", "Rede Neural")

### Agregação de preditores

In [ ]:
%%R
# Regressão da variável de resultado com os preditores de cada método: OLS
ens  <- lm(y.test~ yhat.lm+ yhat.rlasso+ yhat.elnet + yhat.rf+ yhat.pt +yhat.boost)
# # Regressão da variável de resultado com os preditores de cada método: Lasso, post=FALSE
ens2 <- rlasso(y.test~ yhat.lm+ yhat.rlasso+ yhat.elnet + yhat.rf+ yhat.pt + yhat.boost, post=FALSE)

# EQM para aprendizagem agregada
MSE.ens1  <- summary(lm((y.test-ens$fitted.values)^2~1))$coef[1:2]
MSE.ens2  <- summary(lm((y.test-predict(ens2))^2~1))$coef[1:2]

In [ ]:
%%R
# Tabela de resultados de aprendizagem agregada ("ensemble learning")
table2<- matrix(0, 7, 2)

table2[1,1]  <- ens$coefficients[1]
table2[2,1]  <- ens$coefficients[2]
table2[3,1]  <- ens$coefficients[3]
table2[4,1]  <- ens$coefficients[4]
table2[5,1]  <- ens$coefficients[5]
table2[6,1]  <- ens$coefficients[6]
table2[7,1]  <- ens$coefficients[7]


table2[1,2]  <- ens2$coefficients[1]
table2[2,2]  <- ens2$coefficients[2]
table2[3,2]  <- ens2$coefficients[3]
table2[4,2]  <- ens2$coefficients[4]
table2[5,2]  <- ens2$coefficients[5]
table2[6,2]  <- ens2$coefficients[6]
table2[7,2]  <- ens2$coefficients[7]

# Atribuir nomes a columnas e filas
colnames(table2)<- c("Coeficiente (OLS)", "Coeficiente (Lasso)")
rownames(table2)<- c("Constante","OLS-simples","Lasso","Elastic Net com VC", "Random Forest", "Árvore podada","Boosted Trees")

## Resultados

In [ ]:
%%R
# Mostrar resultados
print(table, digits=3)
print(table2, digits=3)

                                EQM Erro Est. do EQM     R^2
Mínimos Quadrados             0.293           0.0243  0.3027
Mínimos Quadrados (flexível)  1.159           0.2382 -1.7547
Lasso                         0.301           0.0241  0.2844
Lasso (flexível)              0.299           0.0240  0.2900
Pós-Lasso                     0.304           0.0242  0.2780
Pós-Lasso (flexível)          0.305           0.0240  0.2753
Lasso com VC                  0.306           0.0241  0.2717
Lasso com VC (flexível)       0.308           0.0240  0.2680
Ridge com VC                  0.303           0.0241  0.2793
Ridge com VC (flexível)       0.405           0.0241  0.0381
Elastic Net com VC            0.307           0.0241  0.2693
Elastic Net com VC (Flexível) 0.309           0.0240  0.2661
Random Forest                 0.292           0.0242  0.3065
Boosted Trees                 0.296           0.0242  0.2967
Árvore podada                 0.340           0.0248  0.1915
Rede Neural             